Downloading and verifying the Dataset:

In [ ]:
import kagglehub
kagglehub.dataset_download('sadmanhsakib/hc18-preprocessed-dataset')

import os
print(os.listdir("/kaggle/input"))

Binary Mask Value Remapping:

In [2]:
from fastai.vision.all import *
import numpy as np
from PIL import Image

class FetalMask(PILMask):
    """Custom PILMask subclass to map mask values of 255 down to class index 1."""
    @classmethod
    def create(cls, fn):
        # Load mask as single-channel grayscale
        img = PILImage.create(fn)
        arr = np.array(img)
        # Threshold: map 255 to 1
        arr = (arr > 127).astype(np.uint8)
        return cls(Image.fromarray(arr))

Calculating the Dice + Focal Combined Loss:
Focal loss focuses on hard-to-classify pixels/boundaries and Dice Loss Directly Optimizes the target metric. Combining both is crucial for medical image segmentation. 

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
        
    def forward(self, logits, targets):
        # Softmax over class dimensions to get probabilities
        probs = F.softmax(logits, dim=1)
        # We target class 1 (fetal head)
        pred = probs[:, 1]
        target = (targets == 1).float()
        
        intersection = (pred * target).sum(dim=(1, 2))
        union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, logits, targets):
        # Cross entropy loss (reduction='none' to scale manually)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1.0 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class DiceFocalLoss(nn.Module):
    """Combined Dice and Focal Loss for robust boundary segmentation."""
    def __init__(self, alpha=0.25, gamma=2.0, smooth=1e-6):
        super().__init__()
        self.dice = DiceLoss(smooth)
        self.focal = FocalLoss(alpha, gamma)
        
    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.focal(logits, targets)